# 02 · Chapter 1: Build your first AI agent

**The idea:** an agent = an LLM + tools + a loop. By the end, the LLM becomes our **senior expert**.

**My plan:**
1. Send my first message to an LLM
2. Measure time, tokens and cost
3. The LLM alone: can it help a customer?
4. Give the LLM tools so it can really *do* things

> Put ONE key in the `.env` file: OpenAI, Anthropic or Gemini (see README).
> No key? It uses the free local model in Ollama instead (slower, less smart, but free).

In [1]:
import sys
sys.path.append("..")   # so Python can find the budget_agent folder

import pandas as pd
pd.set_option("display.max_colwidth", 80)
from dotenv import load_dotenv
load_dotenv("../.env")

from budget_agent.llm import make_llm

llm = make_llm()    # picks the provider from the key in your .env
print("Provider:", llm.provider, "| Model:", llm.model)

Provider: openai | Model: gpt-5.4-mini


## Step 1: My very first message

In [2]:
reply = llm.chat(
    system="You are a helpful assistant.",
    history=[{"role": "user", "content": "In one sentence: what is an AI agent?"}],
)
print(reply.text)

An AI agent is a system that can perceive its environment, make decisions, and take actions autonomously to achieve a goal.


## Step 2: How long did it take, and what did it cost?

In [3]:
reply.stats

CallStats(model='gpt-5.4-mini', input_tokens=26, output_tokens=28, seconds=1.410507000007783, cost_usd=0.0001455, replayed=True)

In [4]:
from budget_agent import config

price_in, price_out = config.PRICES_PER_MILLION.get(llm.model, (0, 0))
print(f"Price: ${price_in} per million input tokens, ${price_out} per million output tokens")
print(f"This one message cost ${reply.stats.cost_usd:.5f} and took {reply.stats.seconds:.1f}s")
print(f"1,000 messages like it would cost about ${reply.stats.cost_usd * 1000:.2f}")

Price: $0.75 per million input tokens, $4.5 per million output tokens
This one message cost $0.00015 and took 1.4s
1,000 messages like it would cost about $0.15


Tiny for one message. But a shop gets thousands every day, and agent messages are much longer than this one.

## Step 3: The LLM alone (version 1 in the code)

I put my prompt and the cost counting into `budget_agent/agent.py` so I don't repeat code.

In [5]:
from budget_agent.agent import SupportAgent

v1 = SupportAgent(use_tools=False, llm=llm)
result = v1.run("where is my order NM10009?")
print(result.reply)
print(f"\n{result.seconds:.1f}s, ${result.cost_usd:.4f}")

Sorry, I can’t access order tracking or customer accounts from here. Please check the tracking link in your order confirmation email, or contact NovaMart support with order ID **NM10009** and your registered phone/email. If you want, I can help you draft a message to support.

1.1s, $0.0003


**Problem:** it's polite, but it cannot see the order. It can only talk.
A real support bot must **look things up and take actions**. That's what tools are for.

## Step 4: The LLM with tools: the senior expert (version 2 in the code)

Tools are plain Python functions in `budget_agent/tools.py`. The LLM can't run them itself. It *asks* us, we run them, and we send back the result.

```
message → LLM → "please run lookup_order(NM10009)" → we run it → result → LLM → reply
```

In [6]:
from budget_agent.tools import TOOL_DEFINITIONS

[tool["name"] for tool in TOOL_DEFINITIONS]

['lookup_order',
 'get_policy',
 'start_refund',
 'cancel_order',
 'update_address',
 'escalate_to_human']

### Writing the agent loop myself

Before using the ready-made agent, I want to see the loop with my own eyes. It's only a few lines:

In [7]:
from budget_agent.agent import SYSTEM_PROMPT
from budget_agent.tools import Shop

shop = Shop()   # our pretend NovaMart database
history = [{"role": "user", "content": "where is my order NM10009?"}]

for step in range(8):                          # never loop forever
    reply = llm.chat(SYSTEM_PROMPT, history, tools=TOOL_DEFINITIONS)

    if reply.stop != "tool_use":               # a plain answer: we're done
        print("💬", reply.text)
        break

    # The LLM asked for tools. Run EVERY one, then send all the results back.
    history.append({"role": "assistant", "reply": reply})
    results = []
    for call in reply.tool_calls:
        output = shop.run_tool(call.name, call.input)
        print(f"🔧 {call.name}({call.input}) → {output[:100]}")
        results.append((call.id, output))
    history.append({"role": "tool_results", "results": results})

🔧 lookup_order({'order_id': 'NM10009'}) → {"order_id": "NM10009", "customer_name": "Rohan Das", "city": "Bengaluru", "product": "Running Shoes
💬 Your order NM10009 is shipped and expected to be delivered on 08 Oct 2026. If you want, I can also help with anything else about the order.


That's the whole agent: **ask the LLM → run the tools it asks for → send the results back → repeat.**

`SupportAgent.run()` in `budget_agent/agent.py` is this same loop. It also adds up the cost and handles a refused request. From here on I'll use it.

In [8]:
v2 = SupportAgent(use_tools=True, llm=llm)

def show(result):
    for name, tool_input, output in result.tool_calls:
        print(f"🔧 {name}({tool_input})\n   → {output[:150]}")
    print(f"\n💬 {result.reply}")
    print(f"\n⏱  {result.seconds:.1f}s   💲 ${result.cost_usd:.4f}   LLM calls: {len(result.calls)}")

show(v2.run("where is my order NM10009?"))

🔧 lookup_order({'order_id': 'NM10009'})
   → {"order_id": "NM10009", "customer_name": "Rohan Das", "city": "Bengaluru", "product": "Running Shoes", "quantity": 1, "amount_inr": 3299, "order_date"

💬 Your order NM10009 is shipped and expected to be delivered on 08 Oct 2026. If you want, I can also help with anything else about the order.

⏱  2.1s   💲 $0.0011   LLM calls: 2


Now it actually looked up the order: running shoes, shipped, arriving 8 Oct. Let me try harder things.

I'll use the same example messages as the README and the app. They live in `config.DEMO_MESSAGES`.

In [9]:
from budget_agent import config

# A refund the expert can do: NM10002 is a ₹5,999 smart watch, delivered 9 days ago
show(v2.run(config.DEMO_MESSAGES["cloud_llm"]))

🔧 lookup_order({'order_id': 'NM10002'})
   → {"order_id": "NM10002", "customer_name": "Aarav Rao", "city": "Pune", "product": "Smart Watch", "quantity": 1, "amount_inr": 5999, "order_date": "2026
🔧 get_policy({})
   → {"policy": "# NovaMart Customer Policy\n\nNovaMart is a pretend online shop used for this workshop.\n\n## Refunds\n- You can ask for a refund within *
🔧 start_refund({'order_id': 'NM10002', 'reason': 'Arrived broken'})
   → {"ok": true, "message": "Refund of ₹5,999 started. It arrives in 5-7 working days."}

💬 Sorry about that — I’ve started the refund for order NM10002. The ₹5,999 should go back to your original payment method in 5–7 working days.

⏱  4.1s   💲 $0.0029   LLM calls: 4


In [10]:
# An angry customer → should go to a human
show(v2.run("this is the 3rd time my delivery is late. I want to talk to a manager NOW"))

🔧 escalate_to_human({'reason': 'Customer reports repeated late delivery and is requesting a manager now.', 'priority': 'high'})
   → {"ok": true, "ticket": "T0002", "message": "A human agent will reply within 2 hours."}

💬 I’m sorry about the repeated delay. I’ve escalated this to a human manager now; ticket T0002 is open and someone should reply within 2 hours.

⏱  2.5s   💲 $0.0011   LLM calls: 2


In [11]:
# Guardrail test: NM10018 is a ₹12,999 chair. Above ₹10,000, the manager (a human) must approve.
show(v2.run(config.DEMO_MESSAGES["human"]))

🔧 lookup_order({'order_id': 'NM10018'})
   → {"order_id": "NM10018", "customer_name": "Sneha Mehta", "city": "Mumbai", "product": "Office Chair", "quantity": 1, "amount_inr": 12999, "order_date":
🔧 get_policy({})
   → {"policy": "# NovaMart Customer Policy\n\nNovaMart is a pretend online shop used for this workshop.\n\n## Refunds\n- You can ask for a refund within *
🔧 escalate_to_human({'reason': 'Refund amount is above ₹10,000 and requires human approval.', 'priority': 'normal'})
   → {"ok": true, "ticket": "T0003", "message": "A human agent will reply within 2 hours."}

💬 I can help with that, but this refund needs human approval because the order amount is above ₹10,000. I’ve escalated it and a human agent will reply within 2 hours.

⏱  4.0s   💲 $0.0029   LLM calls: 4


In [12]:
# Everything the agent changed in the shop
v2.shop.actions

[{'action': 'start_refund', 'order_id': 'NM10002', 'reason': 'Arrived broken'},
 {'action': 'escalate_to_human',
  'ticket': 'T0002',
  'reason': 'Customer reports repeated late delivery and is requesting a manager now.',
  'priority': 'high'},
 {'action': 'escalate_to_human',
  'ticket': 'T0003',
  'reason': 'Refund amount is above ₹10,000 and requires human approval.',
  'priority': 'normal'}]

## What I learned
- The LLM alone can talk but can't act. With tools it's a real agent, our senior expert: it looks things up, takes actions, and follows rules.
- The guardrail (big refunds need the manager) is in **my code**, not in the prompt. The LLM can't skip it.
- **But:** every message goes to the expert, often 2-3 LLM calls. Even "what payment methods do you accept?" costs money and takes seconds. Like a shop where the most expensive expert also answers the phone.

Next, chapter 2: can a tiny ML model take the easy questions? → `03_llm_labels_ml_learns.ipynb`